# Spark:  classic spam classification example using MLLib naive Bayes

refer to:
https://orbifold.net/default/classic-spam-classification-using-spark-mllib/

Unfortunately, this link is broken now!

This is a classic spam classification example using MLLib naive Bayes. 

We use SMS Spam Collection data set. The SMS Spam Collection is a set of SMS tagged messages that have been collected for SMS Spam research. It contains one set of SMS messages in English of 5,574 messages, tagged acording being ham (legitimate) or spam.


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('nlp').getOrCreate()
df = spark.read.csv('./data/SMSSpamCollection', inferSchema = True, sep = '\t')
df.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)



The CSV does not have a header, hence the ugly names. Let’s change this:

In [2]:
df = df.withColumnRenamed('_c0', 'class').withColumnRenamed('_c1', 'text')
df.show()

+-----+--------------------+
|class|                text|
+-----+--------------------+
|  ham|Go until jurong p...|
|  ham|Ok lar... Joking ...|
| spam|Free entry in 2 a...|
|  ham|U dun say so earl...|
|  ham|Nah I don't think...|
| spam|FreeMsg Hey there...|
|  ham|Even my brother i...|
|  ham|As per your reque...|
| spam|WINNER!! As a val...|
| spam|Had your mobile 1...|
|  ham|I'm gonna be home...|
| spam|SIX chances to wi...|
| spam|URGENT! You have ...|
|  ham|I've been searchi...|
|  ham|I HAVE A DATE ON ...|
| spam|XXXMobileMovieClu...|
|  ham|Oh k...i'm watchi...|
|  ham|Eh u remember how...|
|  ham|Fine if thats th...|
| spam|England v Macedon...|
+-----+--------------------+
only showing top 20 rows



Looking at the length of the text we can see that spam is on average longer than ham:

In [3]:
from pyspark.sql.functions import length

df = df.withColumn('length', length(df['text']))
df.show(10)
df.groupBy('class').mean().show()

+-----+--------------------+------+
|class|                text|length|
+-----+--------------------+------+
|  ham|Go until jurong p...|   111|
|  ham|Ok lar... Joking ...|    29|
| spam|Free entry in 2 a...|   155|
|  ham|U dun say so earl...|    49|
|  ham|Nah I don't think...|    61|
| spam|FreeMsg Hey there...|   147|
|  ham|Even my brother i...|    77|
|  ham|As per your reque...|   160|
| spam|WINNER!! As a val...|   157|
| spam|Had your mobile 1...|   154|
+-----+--------------------+------+
only showing top 10 rows

+-----+-----------------+
|class|      avg(length)|
+-----+-----------------+
|  ham|71.45431945307645|
| spam|138.6706827309237|
+-----+-----------------+



So, we’ll take the length into account when setting up the learning pipeline.
First, we need to clean the text a bit:

In [4]:
from pyspark.ml.feature import (CountVectorizer, Tokenizer, 
                                StopWordsRemover, IDF, StringIndexer)
tokenizer = Tokenizer(inputCol = 'text', outputCol = 'token_text')
stop_remove = StopWordsRemover(inputCol = 'token_text', outputCol = 'stop_token')
count_vec = CountVectorizer(inputCol = 'stop_token', outputCol = 'c_vec')
idf = IDF(inputCol = 'c_vec', outputCol = 'tf_idf')
ham_spam_to_numeric = StringIndexer(inputCol = 'class', outputCol = 'label')

The VectorAssembler is the way you hand over the features to a ML algorithm:

In [5]:
from pyspark.ml.feature import VectorAssembler

clean_up = VectorAssembler(inputCols = ['tf_idf', 'length'], outputCol = 'features')

The frame with the transformed fields is thus

In [6]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[ham_spam_to_numeric, tokenizer, stop_remove, count_vec, idf, clean_up])
cleaner = pipeline.fit(df)
clean_df = cleaner.transform(df)
#clean_df = clean_df.select('label', 'features')
clean_df.show(3)

+-----+--------------------+------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+
|class|                text|length|label|          token_text|          stop_token|               c_vec|              tf_idf|            features|
+-----+--------------------+------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+
|  ham|Go until jurong p...|   111|  0.0|[go, until, juron...|[go, jurong, poin...|(13423,[7,11,31,6...|(13423,[7,11,31,6...|(13424,[7,11,31,6...|
|  ham|Ok lar... Joking ...|    29|  0.0|[ok, lar..., joki...|[ok, lar..., joki...|(13423,[0,24,297,...|(13423,[0,24,297,...|(13424,[0,24,297,...|
| spam|Free entry in 2 a...|   155|  1.0|[free, entry, in,...|[free, entry, 2, ...|(13423,[2,13,19,3...|(13423,[2,13,19,3...|(13424,[2,13,19,3...|
+-----+--------------------+------+-----+--------------------+--------------------+--------------------+--------------

We need only two fields for the training

In [7]:
clean_df = clean_df.select('label', 'features')
clean_df.show(3)

+-----+--------------------+
|label|            features|
+-----+--------------------+
|  0.0|(13424,[7,11,31,6...|
|  0.0|(13424,[0,24,297,...|
|  1.0|(13424,[2,13,19,3...|
+-----+--------------------+
only showing top 3 rows



The naive Bayes algorithm is an easy and often used approach for spam classification

In [8]:
from pyspark.ml.classification import NaiveBayes

nb = NaiveBayes()

Since we don’t have a separate validation frame we’ll split the given training set

In [9]:
train, test = clean_df.randomSplit([0.8, 0.2])

Let’s see what the training and testing gives

In [10]:
spam_detector = nb.fit(train)
predictions = spam_detector.transform(test)
predictions.show(3)

+-----+--------------------+--------------------+--------------------+----------+
|label|            features|       rawPrediction|         probability|prediction|
+-----+--------------------+--------------------+--------------------+----------+
|  0.0|(13424,[0,1,2,7,8...|[-796.15058621537...|[1.0,2.4271720089...|       0.0|
|  0.0|(13424,[0,1,4,50,...|[-834.97444142082...|[1.0,1.3388018897...|       0.0|
|  0.0|(13424,[0,1,7,8,1...|[-884.47702061624...|[1.0,1.0384127245...|       0.0|
+-----+--------------------+--------------------+--------------------+----------+
only showing top 3 rows



The accuracy can be obtained like so

In [11]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator()
print("Test Accuracy: " + str(evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})))

Test Accuracy: 0.9125338142470695


In [12]:
spark.stop()

Question:
How to improve the accuracy of this spam classfication method using NB?
here is a link may give you some general ideas:
https://stackoverflow.com/questions/3473612/ways-to-improve-the-accuracy-of-a-naive-bayes-classifier